# 01 — Load Signal Snapshots

Pulls `signal_snapshots` for `sum_arb` strategy from micro-arb's Railway Postgres.

**Invariants**: V1 (spread = yes+no-1 check), C1 (no arb logic here), V9 (config from file).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config, db

cfg = config.load()
print('Config:', cfg)

In [ ]:
# Pull all signal_snapshots within the backtest window.
# signal_snapshots stores poly_yes_price + poly_no_price so V1 can be verified.
# No strategy column on signal_snapshots; sum_arb signals identified by
# presence of both poly_yes_price and poly_no_price (NULL for momentum path).
rows = db.query("""
    SELECT
        correlation_id,
        contract_id,
        direction,
        implied_prob_market   AS yes_price,
        implied_prob_real     AS implied_no_fair,
        poly_yes_price,
        poly_no_price,
        spread,
        expected_edge,
        position_size_usd,
        decision,
        rejection_reason,
        snapshot_time
    FROM signal_snapshots
    WHERE poly_yes_price IS NOT NULL
      AND poly_no_price  IS NOT NULL
    ORDER BY snapshot_time ASC
""")

signals = pd.DataFrame(rows)
print(f'Rows: {len(signals)}')
signals.head()

In [ ]:
# V1: verify spread = poly_yes_price + poly_no_price - 1
signals['spread_check'] = signals['poly_yes_price'] + signals['poly_no_price'] - 1
drift = (signals['spread'].astype(float) - signals['spread_check']).abs()
bad = drift > 1e-6
print(f'V1 violations (|drift| > 1e-6): {bad.sum()} / {len(signals)}')
assert bad.sum() == 0, f'V1 FAIL: {bad.sum()} rows have spread mismatch'
print('V1 OK')

In [ ]:
# Decision breakdown
print(signals.groupby(['decision', 'rejection_reason']).size().rename('count').to_string())